In [1]:
#!pip install sympy

In [2]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import extract_qhd_solution_vector
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [3]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [12]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 3 # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 3 buses 3 lines and 2 generators.


In [ ]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 1000
alpha = 1   # 对偶步长尽量小一点
max_outer = 60
tol = 1e-4
option = 2 # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # openjij / simbi

# ========= 新增：初始化日志文件 =========
log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=8,
                agents=1024,
                max_steps=5000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                #ballistic=True,
                #heated=True,
                best_only=False,
                verbose=True
            )
        else:
            qhd_model.openjij_setup(
                resolution=6,
                shots=2048,
                sampler_name="SQASampler",
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 10000,
                    "reinitialize_state": True,
                },
            )
        response = qhd_model.optimize(refine=True, verbose=0)
        #response = qhd_model.optimize(refine=False, verbose=0)

        x_new = extract_qhd_solution_vector(response, expected_len=len(variable_list))

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=alpha,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    # Keep rho fixed for stability diagnostics.
    print("Keeping rho fixed at", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(x_new)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    log_file = PrintQHDACOPFResults(
        model,
        x_new,
        log_file=log_file,
        iteration=k,
        folder="logs",
        print_to_console=True,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-3_04-15-2026_20-13-02.txt

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---
||h(x)|| = 4.062522915230681
[rho-check] ||h_old||=5.181e-01, ||h_new||=4.063e+00, rho=1e+03
Keeping rho fixed at 1000
Constraint check: False
Update time: 2026-04-15 20:13:04
Iteration: 0
rho: 1000
alpha: 0.0005
objective_value: 0
feasible: False
max_abs_h: 2.988951469775e+00
l2_norm_h: 4.062522915231e+00
lambda_inf_norm: 1.494475734888e-03
lambda_l2_norm: 2.031261457615e-03
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.000001	-0.000000	1.000001	0.000001	1.937198	0.000000	0.000000
2	0.905942	0.038612	0.900004	0.283126	-1.989001	0.000000	0.000000
3	0.937311	-0.007168	0.935212	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1	1	2	-0.112736	0.102385	1.647044	-1.725824	-0.010351	-0.078780
2	1	3	0.114497	

In [14]:
Lag

0.00375*P_G0**2 + 1.99994796883275*P_G0 + 0.0175*P_G1**2 + 1.74989836069678*P_G1 - 0.000570031434387314*P_ij0 + 0.000559497487608434*P_ij1 + 1.85277930448758e-5*P_ij2 - 2.49533626459316e-5*P_ij3 + 2.73374573688927e-5*P_ij4 - 2.87121921245328e-5*P_ij5 - 6.43648336846111e-10*Q_G0 - 0.000177525183479788*Q_G1 - 0.000322966329258916*Q_ij0 - 6.19813470939462e-5*Q_ij1 + 1.68268034229083e-6*Q_ij2 - 2.3014775911341e-5*Q_ij3 - 2.4124191494839e-5*Q_ij4 + 6.28842526854827e-6*Q_ij5 - 0.00230443260904932*S_tot_sq0 - 0.00245890299729006*S_tot_sq1 - 7.35897077596945e-5*S_tot_sq2 - 8.5333142076342e-5*S_tot_sq3 - 7.65204397976441e-5*S_tot_sq4 - 6.78261889570821e-5*S_tot_sq5 + 200000.0*V_I0**2 + 7.38498537748628e-5*V_I0 - 0.000224338856693404*V_I1 + 0.000221315993850818*V_I2 - 0.00578002780493648*V_R0 + 0.00690904790628998*V_R1 - 0.00109010864533337*V_R2 + 3.29385911879143e-11*V_sq0 - 9.5692514155793e-6*V_sq1 - 2.89789286772052e-6*V_sq2 + 500.0*(20.0*V_R0 - 20.0)**2 + 500.0*(0.247324754926013*P_ij0 + 0.1

In [7]:
model.P_D

array([0. , 0.3])